# Pruebas paso a paso del chatbot

Ejecuta las celdas en orden desde `backend/`, con el entorno Conda o virtual activado. Completa `.env` antes de iniciar la API. La clave Gemini es opcional: sin ella se prueba el modo local.

## 1. Comprobar configuración y dependencias

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()
print('DB_HOST:', os.getenv('DB_HOST', 'localhost'))
print('DB_NAME:', os.getenv('DB_NAME', 'atencion_db'))
print('Gemini configurado:', bool(os.getenv('GOOGLE_API_KEY')))


DB_HOST: localhost
DB_NAME: atencion_db
Gemini configurado: True


## 2. Conectar con MySQL y validar los datos cargados

In [2]:
import mysql.connector
conn = mysql.connector.connect(host=os.getenv('DB_HOST','localhost'), port=int(os.getenv('DB_PORT','3306')), user=os.getenv('DB_USER','root'), password=os.getenv('DB_PASSWORD',''), database=os.getenv('DB_NAME','atencion_db'))
cur = conn.cursor()
cur.execute('SELECT COUNT(*) FROM clientes'); print('Clientes:', cur.fetchone()[0])
cur.execute('SELECT COUNT(*) FROM atencion_cliente'); print('Consultas:', cur.fetchone()[0])
cur.close(); conn.close()


Clientes: 50
Consultas: 100


## 3. Consultar clientes disponibles

In [3]:
conn = mysql.connector.connect(host=os.getenv('DB_HOST','localhost'), port=int(os.getenv('DB_PORT','3306')), user=os.getenv('DB_USER','root'), password=os.getenv('DB_PASSWORD',''), database=os.getenv('DB_NAME','atencion_db'))
cur = conn.cursor(dictionary=True)
cur.execute('SELECT id, nombre, apellidos, plan FROM clientes WHERE activo=1 ORDER BY id LIMIT 10')
clientes = cur.fetchall()
clientes


[{'id': 1,
  'nombre': 'Lucía',
  'apellidos': 'Sánchez Álvarez',
  'plan': 'premium'},
 {'id': 2, 'nombre': 'Carlos', 'apellidos': 'López Sánchez', 'plan': 'basico'},
 {'id': 3, 'nombre': 'María', 'apellidos': 'Rubio Álvarez', 'plan': 'basico'},
 {'id': 4, 'nombre': 'Javier', 'apellidos': 'Cortés Ramos', 'plan': 'empresa'},
 {'id': 5, 'nombre': 'Ana', 'apellidos': 'Vázquez Vázquez', 'plan': 'basico'},
 {'id': 6, 'nombre': 'David', 'apellidos': 'Molina Ortega', 'plan': 'premium'},
 {'id': 7, 'nombre': 'Laura', 'apellidos': 'Díaz García', 'plan': 'basico'},
 {'id': 8,
  'nombre': 'Pablo',
  'apellidos': 'Iglesias Torres',
  'plan': 'basico'},
 {'id': 9, 'nombre': 'Carmen', 'apellidos': 'Molina Díaz', 'plan': 'basico'},
 {'id': 10,
  'nombre': 'Sergio',
  'apellidos': 'Blanco Iglesias',
  'plan': 'basico'}]

## 4. Levantar la API

En otra terminal, desde `backend/`, ejecuta `uvicorn main:app --reload`. Deja ese proceso activo.

In [5]:
import requests
BASE_URL = 'http://127.0.0.1:8000'
requests.get(f'{BASE_URL}/api/health').json()


{'status': 'ok'}

## 5. Probar lista de clientes y crear una conversación

In [9]:
clientes_api = requests.get(f'{BASE_URL}/api/clients').json()
print('Clientes activos:', len(clientes_api))
cliente_id = clientes_api[0]['id']
primera = requests.post(f'{BASE_URL}/api/chat', json={'client_id': cliente_id, 'message': '¿Cómo puedo cambiar mi correo?'})
primera.status_code, primera.json()


Clientes activos: 47


(200,
 {'session_id': '93ce2e70-6910-446e-8f26-6e5c7e406f89',
  'answer': "[{'type': 'text', 'text': 'Hola, Lucía. No dispongo de la información necesaria para indicarte el procedimiento para cambiar tu correo electrónico. Te recomiendo ponerte en contacto con un agente para que pueda ayudarte a gestionar este cambio en tu cuenta.', 'extras': {'signature': 'EoIQCv8PAWkUfRPt1LGsDjgg2RWM3g1uaGvbw/bHOMLdy3zvtvGuCCS1cXHbDcYakCSfpiAn7etMrOZSxq0k/S/S8Dps/bD41A+j4t7kbT1w6nZDdXaMHaxk9wpxiV5xa3EuPDMdz/8ir/yKaeManIOVBRqMHS7X29s85dGqCOZuRDjo3KJ+Dd0EB3SE1MnpIw03wpTEd5OdUkdamomITKVzENUm2w3Ku6H7QuXOQ4C16dyx4EuDgEvEfgat31JjkgkK7o9sAPHHxfOw2S+WaNq6p910HgYLA+NnhcbTNPSmtoDM/mdEr2twoL7an6Rsqpd6PmB7niFK7PzLSHUH/A6BwlFfaw2dx1h0xArpUKn78YMCBFnvlduRH8FdDRSETmeCmjDsqzuASixO6HFJ00LUWp/qpzN2//xjwy4IEjabHc5qja1MqR4aBbqd/CJ48HkmsWnzIfly2M3/LLwhWwADNwrsGjEYu4MM1v+1wOZyLgw9HYUKUmE0XMOn6006HTJ2YzynoV/WdI8l4M7HSsRjccW+P5InBysOpUHHfrq2HBxWVJYLQEosIao6cUJ807w24gl1B9++kyDtv+/FFI14mixVESzN4q+b4DlkixoalG2X1wlEWYBG5kxXP6+N

## 6. Continuar la sesión y comprobar memoria

La sesión conserva los últimos turnos en memoria del proceso de API.

In [10]:
sesion = primera.json()['session_id']
segunda = requests.post(f'{BASE_URL}/api/chat', json={'client_id': cliente_id, 'message': '¿Dónde encuentro esa opción?', 'session_id': sesion})
segunda.status_code, segunda.json()


(200,
 {'session_id': '93ce2e70-6910-446e-8f26-6e5c7e406f89',
  'answer': "[{'type': 'text', 'text': 'Hola, Lucía. \\n\\nSi te refieres a la verificación en dos pasos, la encontrarás accediendo a **Mi Cuenta > Seguridad**. \\n\\nSi te refieres a la opción para cambiar tu correo electrónico, no dispongo de esa información en el sistema, por lo que te recomiendo ponerte en contacto con un agente para que te guíe en el proceso.', 'extras': {'signature': 'ErgjCrUjAWkUfRNFI7WL2XOpKtZsMkTttgI2DR7NeJ06IuGJuhOLGV4CBOOpY5yWK3yS2tRzMve520aYb9Cw1kBpDUjVbQY1wKQkCbRdTahvLhuw/qxJg0wt4SPmMm8QB2oSdAFSbJFnHQxlQUMi1QJSVHXDisSzu4e4ghytMvHvzJmn6NDiO1zbmgK/FN8murtFNsEaFJ8NCA8MmwSPtn4cE5c0qwM8eI9PN0FiMJ/7WaBk+Ow8n7tdKd/Z1/ZjWDKSB0RvA5ARC/S0Ryz6I41hRtecd+D6ePiAyN9jPD3azU8ipgDTzTmEZIbPGG4k95dPySLlf6EIEvjLX/5w2kJ1fVLQr7NTZyrpjLa9vjEJOsc6Y83wxmiDVggt166IeId6leyC7/oYcqawGOpixj4crFU4oKnlG16oOwYhXK6z3Xz6H7xtzYC8kC8744U7bJUd8JrULDBRZOsAuB5dfNEnn5gYAe4zTbuk7Kc1ZQhKtkmYDfgfjqCDM0F+SUWL8loC8xUDGo8gvAQfQc6/T6XjN21KR+PV

## 7. Revisar el registro y limpiar sesión

In [11]:
conn = mysql.connector.connect(host=os.getenv('DB_HOST','localhost'), port=int(os.getenv('DB_PORT','3306')), user=os.getenv('DB_USER','root'), password=os.getenv('DB_PASSWORD',''), database=os.getenv('DB_NAME','atencion_db'))
cur = conn.cursor(dictionary=True)
cur.execute('SELECT fecha, canal, consulta, respuesta FROM atencion_cliente WHERE cliente_id=%s ORDER BY fecha DESC LIMIT 3', (cliente_id,))
cur.fetchall()


[{'fecha': datetime.datetime(2026, 10, 22, 10, 17, 27),
  'canal': 'chatbot',
  'consulta': '¿Dónde encuentro esa opción?',
  'respuesta': "[{'type': 'text', 'text': 'Hola, Lucía. \\n\\nSi te refieres a la verificación en dos pasos, la encontrarás accediendo a **Mi Cuenta > Seguridad**. \\n\\nSi te refieres a la opción para cambiar tu correo electrónico, no dispongo de esa información en el sistema, por lo que te recomiendo ponerte en contacto con un agente para que te guíe en el proceso.', 'extras': {'signature': 'ErgjCrUjAWkUfRNFI7WL2XOpKtZsMkTttgI2DR7NeJ06IuGJuhOLGV4CBOOpY5yWK3yS2tRzMve520aYb9Cw1kBpDUjVbQY1wKQkCbRdTahvLhuw/qxJg0wt4SPmMm8QB2oSdAFSbJFnHQxlQUMi1QJSVHXDisSzu4e4ghytMvHvzJmn6NDiO1zbmgK/FN8murtFNsEaFJ8NCA8MmwSPtn4cE5c0qwM8eI9PN0FiMJ/7WaBk+Ow8n7tdKd/Z1/ZjWDKSB0RvA5ARC/S0Ryz6I41hRtecd+D6ePiAyN9jPD3azU8ipgDTzTmEZIbPGG4k95dPySLlf6EIEvjLX/5w2kJ1fVLQr7NTZyrpjLa9vjEJOsc6Y83wxmiDVggt166IeId6leyC7/oYcqawGOpixj4crFU4oKnlG16oOwYhXK6z3Xz6H7xtzYC8kC8744U7bJUd8JrULDBRZOsAuB5dfNEnn5gYAe4

In [12]:
requests.delete(f"{BASE_URL}/api/sessions/{sesion}").json()


{'status': 'cleared'}

## Solución de problemas

- `Can't connect to MySQL`: revisa que MySQL esté activo, el puerto y las credenciales del `.env`.
- `Unknown database`: importa `database/schema.sql` desde la raíz.
- Error de conexión desde el navegador: confirma que uvicorn está activo y `CORS_ORIGINS` contiene el origen del frontend.
- Gemini no disponible: confirma `GOOGLE_API_KEY`; sin clave, el endpoint usa una respuesta local de respaldo.